# Incluye el M02 03 y 04
# M02-01 Ingesta CSV y Json


## Paso 1 - Arranque y sesion



In [30]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)

spark = get_spark("novashop-m02")
print(RAW.exists())


ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated
True


## Paso 2 - CSV de clientes y pedidos.

Leer ahora los datos de CSV y de los ficheros de clientes y pedidos a partir del directorio raw en donde se encuentran los ficheros.



In [31]:
customers = spark.read.option("header", True).csv(str(RAW / "customers.csv"))
orders = spark.read.option("header", True).csv(str(RAW / "orders.csv"))
print("customers", customers.count(), "orders", orders.count())
orders.printSchema()
orders.show(3, truncate=False)

customers 250 orders 800
root
 |-- OrderId: string (nullable = true)
 |-- CustomerId: string (nullable = true)
 |-- OrderDate: string (nullable = true)
 |-- Status: string (nullable = true)
 |-- Channel: string (nullable = true)

+-------+----------+-------------------+------+-------+
|OrderId|CustomerId|OrderDate          |Status|Channel|
+-------+----------+-------------------+------+-------+
|O00001 |NULL      |2024-04-16 20:00:00|paid  |app    |
|O00002 |NULL      |2024-12-04 02:00:00|paid  |store  |
|O00003 |NULL      |2024-03-29 17:00:00|paid  |WEB    |
+-------+----------+-------------------+------+-------+
only showing top 3 rows



## Paso 3 - Json array y JSON 

Ahora cargamos el json de productos y eventos. 

In [32]:
products = spark.read.option("multiLine", True).json(str(RAW / "products.json"))
events = spark.read.json(str(RAW / "events.jsonl"))
print("products", products.count(), "events", events.count())
products.printSchema()
events.printSchema()

products 60 events 2500
root
 |-- category: string (nullable = true)
 |-- listPrice: string (nullable = true)
 |-- name: string (nullable = true)
 |-- productId: string (nullable = true)

root
 |-- customer_id: string (nullable = true)
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- page: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- session_id: string (nullable = true)
 |-- ts: string (nullable = true)



## Mejora añadida

Ahora leemos el fichero orde_items.csv con cabecera incluida, mostrar el conteo y las tres primera filas.


In [33]:
order_items = spark.read.option("header", True).csv(str(RAW / "order_items.csv"))
record_order_items = order_items.count()
print ("Registros de order items ", record_order_items)
order_items.show()

Registros de order items  2046
+--------+----------+---+----------+--------+
|order_id|product_id|qty|unit_price|discount|
+--------+----------+---+----------+--------+
|  O00001|      P053|  3|    114.41|    0.00|
|  O00002|      P016|  1|     90.86|    0.20|
|  O00002|      P007|  1|     16.79|    0.05|
|  O00003|      P011|  4|    125.80|    0.05|
|  O00003|      P023|  5|     96.95|    0.20|
|  O00003|      P031|  5|    164.48|    0.15|
|  O00003|      P015|  2|    115.56|    0.20|
|  O00004|      P019|  1|     83.08|    0.00|
|  O00005|      P022|  2|    222.53|    0.00|
|  O00006|      P036|  1|     87.79|    0.10|
|  O00007|      P046|  1|    110.66|    0.15|
|  O00007|      P011|  5|    222.98|    0.00|
|  O00007|      P004|  1|     86.11|    0.00|
|  O00007|      P027|  5|    181.89|    0.15|
|  O00008|      P043|  3|     77.85|    0.00|
|  O00008|      P004|  4|    145.07|    0.20|
|  O00008|      P032|  2|     18.34|    0.00|
|  O00009|      P006|  3|    206.84|    0.15|
|  

## Comenzamos con en M02-03 Schema - tipos.

## Paso 2 Realizamos el camibo de camelCase a snake_case


In [34]:
from pyspark.sql.types import StructType, StructField, StringType

orders_raw_schema = StructType([
    StructField("OrderId", StringType(), True),
    StructField("CustomerId", StringType(), True),
    StructField("OrderDate", StringType(), True),
    StructField("Status", StringType(), True),
    StructField("Channel", StringType(), True),
])
orders = (
    spark.read.option("header", True)
    .schema(orders_raw_schema)
    .csv(str(RAW / "orders.csv"))
    .withColumnRenamed("OrderId", "order_id")
    .withColumnRenamed("CustomerId", "customer_id")
    .withColumnRenamed("OrderDate", "order_ts_raw")
    .withColumnRenamed("Status", "status")
    .withColumnRenamed("Channel", "channel")
)
orders.printSchema()
print(orders.count())

root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_ts_raw: string (nullable = true)
 |-- status: string (nullable = true)
 |-- channel: string (nullable = true)

800


## Paso 03 - Timestamp con varios formatos



In [35]:
from pyspark.sql.functions import col, coalesce, to_timestamp

orders = orders.withColumn(
    "order_ts",
    coalesce(
        to_timestamp(col("order_ts_raw"), "yyyy-MM-dd HH:mm:ss"),
        to_timestamp(col("order_ts_raw"), "dd/MM/yyyy"),
    ),
).drop("order_ts_raw")
print("nulos de fecha", orders.where(col("order_ts").isNull()).count())
orders.printSchema()

nulos de fecha 0
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- status: string (nullable = true)
 |-- channel: string (nullable = true)
 |-- order_ts: timestamp (nullable = true)



## Paso 4 - Líneas de enteros y decimales

En el CSV la columna unit_price es un string. Se trata de cambiar el tipo de dato para que admite enteros.

Y por ulitmo comprobar que tipos tenemos obteniendo el esquema y una muestra del mismo.

In [36]:
from pyspark.sql.types import IntegerType, DecimalType

items = (
    spark.read.option("header", True).csv(str(RAW / "order_items.csv"))
    .withColumn("qty", col("qty").cast(IntegerType()))
    .withColumn("unit_price", col("unit_price").cast(DecimalType(10, 2)))
    .withColumn("discount", col("discount").cast(DecimalType(5, 2)))
)
items.printSchema()
items.select("unit_price").limit(3).show()

items.limit(4).show()


root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- unit_price: decimal(10,2) (nullable = true)
 |-- discount: decimal(5,2) (nullable = true)

+----------+
|unit_price|
+----------+
|    114.41|
|     90.86|
|     16.79|
+----------+

+--------+----------+---+----------+--------+
|order_id|product_id|qty|unit_price|discount|
+--------+----------+---+----------+--------+
|  O00001|      P053|  3|    114.41|    0.00|
|  O00002|      P016|  1|     90.86|    0.20|
|  O00002|      P007|  1|     16.79|    0.05|
|  O00003|      P011|  4|    125.80|    0.05|
+--------+----------+---+----------+--------+



# # Paso 5 - eventos con schema

Ingestar eventos json infiriendo el schema.

In [37]:
from pyspark.sql.types import TimestampType

events_schema = StructType([
    StructField("event_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("ts", TimestampType(), True),
    StructField("session_id", StringType(), True),
    StructField("page", StringType(), True),
    StructField("product_id", StringType(), True),
])
events = spark.read.schema(events_schema).json(str(RAW / "events.jsonl"))
events.printSchema()
events.limit(5).show()
print(events.count())


root
 |-- event_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- ts: timestamp (nullable = true)
 |-- session_id: string (nullable = true)
 |-- page: string (nullable = true)
 |-- product_id: string (nullable = true)

+--------+-----------+----------+-------------------+----------+---------+----------+
|event_id|customer_id|event_type|                 ts|session_id|     page|product_id|
+--------+-----------+----------+-------------------+----------+---------+----------+
| E000001|       NULL| page_view|2024-08-11 23:46:00|     S0806| /catalog|      P038|
| E000002|       NULL| page_view|2024-07-05 11:43:00|     S0636|/checkout|      P058|
| E000003|       NULL| page_view|2024-07-24 22:03:00|     S0055|    /cart|      P009|
| E000004|       NULL| page_view|2024-05-26 16:57:00|     S0474|/checkout|      P030|
| E000005|       NULL| page_view|2024-11-01 13:01:00|     S0057| /product|      P054|
+--------+-----------+--

## Mejora -- Catálogo en snake-case y decimal

Leer el fichero products.json, renombar productId por product-id, listPrice por list-price y transformar list-price a DecimalType(10,2). Tres list_price nulos es correcto (se limpiarán en el lab siguiente)



In [38]:
products = (
    spark.read.option("multiline", True).json(str(RAW / "products.json"))
    .withColumnRenamed("productId", "product_id")
    .withColumnRenamed("listPrice", "list_price")
    .withColumn("list_price", col("list_price").cast(DecimalType(10,2)))
)
products.printSchema()
products.limit(4).show()

root
 |-- category: string (nullable = true)
 |-- list_price: decimal(10,2) (nullable = true)
 |-- name: string (nullable = true)
 |-- product_id: string (nullable = true)

+-----------+----------+------------+----------+
|   category|list_price|        name|product_id|
+-----------+----------+------------+----------+
|     sports|    233.51|Producto 001|      P001|
|      books|    133.48|Producto 002|      P002|
|       home|     84.17|Producto 003|      P003|
|electronics|    250.06|Producto 004|      P004|
+-----------+----------+------------+----------+



## M2-03 Calidad y limpieza

Arranque y prepración del entorno



In [39]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


from pyspark.sql.functions import col, coalesce, to_timestamp, when, trim
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DecimalType, TimestampType

spark = get_spark("novashop-m02")

orders = (
    spark.read.option("header", True).csv(str(RAW / "orders.csv"))
    .withColumnRenamed("OrderId", "order_id")
    .withColumnRenamed("CustomerId", "customer_id")
    .withColumnRenamed("OrderDate", "order_ts_raw")
    .withColumnRenamed("Status", "status")
    .withColumnRenamed("Channel", "channel")
    .withColumn(
        "order_ts",
        coalesce(
            to_timestamp(col("order_ts_raw"), "yyyy-MM-dd HH:mm:ss"),
            to_timestamp(col("order_ts_raw"), "dd/MM/yyyy"),
        ),
    )
    .drop("order_ts_raw")
)
customers = spark.read.option("header", True).csv(str(RAW / "customers.csv"))
products = (
    spark.read.option("multiLine", True).json(str(RAW / "products.json"))
    .withColumnRenamed("productId", "product_id")
    .withColumnRenamed("listPrice", "list_price")
    .withColumn("list_price", col("list_price").cast(DecimalType(10, 2)))
)
items = (
    spark.read.option("header", True).csv(str(RAW / "order_items.csv"))
    .withColumn("qty", col("qty").cast(IntegerType()))
    .withColumn("unit_price", col("unit_price").cast(DecimalType(10, 2)))
    .withColumn("discount", col("discount").cast(DecimalType(5, 2)))
)
events = spark.read.schema(
    StructType([
        StructField("event_id", StringType(), False),
        StructField("customer_id", StringType(), True),
        StructField("event_type", StringType(), True),
        StructField("ts", TimestampType(), True),
        StructField("session_id", StringType(), True),
        StructField("page", StringType(), True),
        StructField("product_id", StringType(), True),
    ])
).json(str(RAW / "events.jsonl"))
print(orders.count(), customers.count(), products.count(), items.count(), events.count())


ROOT    /workspaces/python-pyspark-201
RAW     /workspaces/python-pyspark-201/data/raw existe: True
STAGING /workspaces/python-pyspark-201/data/staging
CURATED /workspaces/python-pyspark-201/data/curated
800 250 60 2046 2500


## Paso 2 - clientes y prodcutos

Cuando el país es vacio es recuperable (UNK) y producto sin precio se tira

In [40]:
customers_clean = customers.withColumn(
    "country",
    when(col("country").isNull() | (trim(col("country")) == ""), "UNK").otherwise(col("country")),
)
products_clean = products.where(col("list_price").isNotNull())
print("customers", customers_clean.count(), "unk", customers_clean.where(col("country") == "UNK").count())
print("products", products_clean.count())


customers 250 unk 5
products 57


## Paso 3 - Pedidos líneas y eventos

Como las claves vacías pueden romper algunos joins, lous huérfanos CX* y P999 se quenda y el M04 los visibiliza

In [41]:
orders_clean = orders.where(trim(col("customer_id")) != "")
items_clean = items.where((trim(col("product_id")) != "") & (col("qty") > 0))
events_clean = events.where(col("customer_id").isNotNull())
print("orders", orders_clean.count())
print("items", items_clean.count())
print("events", events_clean.count())

orders 788
items 2010
events 2420


# Paso 4 - Escribe staging en formato (Parquet)

Parquet guarda el schema, de este modo el siguiete módulo ya no vuelve a inferir el CSV

In [43]:
STAGING.mkdir(parents=True, exist_ok=True)
pairs = {
    "customers_clean": customers_clean,
    "products_clean": products_clean,
    "orders_clean": orders_clean,
    "order_items_clean": items_clean,
    "events_clean": events_clean,
}
for name, frame in pairs.items():
    dest = STAGING / name
    frame.write.mode("overwrite").parquet(str(dest))
    print(name, dest)
print("releer orders", spark.read.parquet(str(STAGING / "orders_clean")).count())


customers_clean /workspaces/python-pyspark-201/data/staging/customers_clean
products_clean /workspaces/python-pyspark-201/data/staging/products_clean
orders_clean /workspaces/python-pyspark-201/data/staging/orders_clean
order_items_clean /workspaces/python-pyspark-201/data/staging/order_items_clean
events_clean /workspaces/python-pyspark-201/data/staging/events_clean
releer orders 788
